# 11 — Retrieval

Modules 08 and 09 answered Helena from a **database**. Module 02 answered Amsterdam from a **CSV**. Those answers have an address: a table, a row, a function.

A return window does not. It lives in a paragraph in `data/corpus/`. You cannot `SELECT` it. You also should not paste all 36 files into the system prompt — module 05 already measured what that costs.

**Retrieval** is the boring, correct move: turn the question into a vector, find the nearest files, put *those* files in the prompt, then ask. That is single-shot RAG. One retrieve, one generate. Module 12 will loop it. Module 14 will show that a retrieved file is still untrusted data.

```mermaid
graph TD
    A[36 files on disk] --> B[Chroma]

    C[Question] -->|Embed question| D[Question embedding]
    D -->|Find nearest k documents| B

    B --> E[Retrieved file content]
    E --> F[Prompt]
    C --> F

    F --> G[LLM]
    G --> H[Answer]
```


## 1. Learn

```
08/09  the fact is in Chinook — query it
02/03  the fact is in a CSV — call a tool
05     the list is a budget
10     the model wrote code
11     you are here — the fact is in a folder of markdown
12     retrieve, assess, retrieve again
14     a retrieved file can carry instructions
```

Three places an answer can live, and RAG is only one of them:

| Question | Where the answer is | What to do |
|---|---|---|
| How many invoices does Helena have? | `chinook.db` | SQL. Module 08. |
| Sydney to Madrid? | `flight_data.csv` | A tool. Module 02. |
| How long do I have to return an unopened CD? | `data/corpus/` | Retrieve, then generate. |
| What is next year's revenue target? | Nowhere in this repo | Say you do not know. |

If you retrieve for Helena's invoice count you will get a policy about invoices, not the number 7. RAG is for unstructured text you cannot otherwise reach. It is not a default.

**Chunking.** These files are already short. Today **one file is one chunk**. That is a decision, not a library. Module 12 can split. We will not.

**The store.** Chroma, local, in this process. OpenAI embeds; Chroma only stores and ranks. LlamaIndex and Pinecone are names on a slide — same retrieval, different packaging. We do not install them. A managed store in 16, if at all.

Two questions in this corpus have **no** answer. They are on purpose. An honest system says so. A chatbot with a vector store invents a mobile number.


## 2. Do

### Load the environment and read one file yourself


In [1]:
from pathlib import Path
import os

from chromadb import Client
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI


load_dotenv(find_dotenv(usecwd=True))
ROOT = Path(find_dotenv(usecwd=True)).parent

api_key = os.environ.get("OPENAI_API_KEY", "").strip()
model = os.environ.get("MODEL_DEFAULT", "").strip()
embed_model = os.environ.get("EMBEDDING_MODEL", "").strip()
assert api_key, "OPENAI_API_KEY is missing."
assert model, "MODEL_DEFAULT is missing from .env."
assert embed_model, "EMBEDDING_MODEL is missing. Copy the line from .env.example."

client = OpenAI()
CORPUS = ROOT / "data" / "corpus"
files = sorted(CORPUS.glob("*.md"))
print("OPENAI_API_KEY is set:", True)
print("MODEL_DEFAULT:", model)
print("EMBEDDING_MODEL:", embed_model)
print("n files:", len(files))
print("first five:", [p.name for p in files[:5]])
print()
print("--- policy_returns.md ---")
print((CORPUS / "policy_returns.md").read_text())


OPENAI_API_KEY is set: True
MODEL_DEFAULT: gpt-5.4-nano
EMBEDDING_MODEL: text-embedding-3-small
n files: 36
first five: ['policy_accounts.md', 'policy_cancellations.md', 'policy_damaged_media.md', 'policy_data_retention.md', 'policy_digital_downloads.md']

--- policy_returns.md ---
# Returns

Physical items (CDs, vinyl, boxed sets) may be returned within 30 days of the invoice date if unopened.

Opened physical media can be returned within 14 days only if the disc is defective. A replacement is preferred over a refund.

Digital downloads and streamed albums cannot be returned. See policy_digital_downloads.md.

Start a return by writing to support. Include the invoice number. Support will reply within the SLA in policy_support_hours.md.



30 days if unopened. 14 if opened and defective. Digital cannot be returned. You did not need a model.

### An embedding is a list of floats

Same API family, different endpoint: `embeddings.create`, not `chat.completions.create`. We print the length and the first eight numbers. The rest is more of the same.


In [2]:
sample = "How long do I have to return an unopened CD?"
emb = client.embeddings.create(model=embed_model, input=sample)
vec = emb.data[0].embedding
print("n dimensions:", len(vec))
print("first 8:     ", [round(x, 5) for x in vec[:8]])
print("prompt_tokens (this embed):", emb.usage.prompt_tokens)


n dimensions: 1536
first 8:      [0.00742, 0.07263, 0.01944, 0.0014, 0.01952, 0.00813, 0.00794, 0.00899]
prompt_tokens (this embed): 12


That is the question, as far as the store is concerned. Nearness is a distance between two of those lists.

### Load the folder

One id per file. The document is the file text. We embed each one ourselves and hand Chroma the vectors, so the store is not a second embedding library.


In [3]:
def embed(text: str) -> list[float]:
    return client.embeddings.create(model=embed_model, input=text).data[0].embedding


ids = []
documents = []
metadatas = []
vectors = []
for path in files:
    text = path.read_text()
    ids.append(path.name)
    documents.append(text)
    metadatas.append({"path": path.name})
    vectors.append(embed(text))

chroma = Client()
collection = chroma.create_collection("corpus")
collection.add(ids=ids, documents=documents, metadatas=metadatas, embeddings=vectors)
print("stored:", collection.count())


stored: 36


36 vectors. Rebuild is cheap at this size, so we keep Chroma in memory. A production store would persist; the shape of `add` / `query` does not change.

### Retrieve, no generate

Nearest three files for the return-window question. Look at **names and distances** before anyone writes a sentence.


In [4]:
def retrieve(question: str, k: int = 3):
    q = embed(question)
    got = collection.query(query_embeddings=[q], n_results=k)
    rows = []
    for i in range(len(got["ids"][0])):
        rows.append(
            {
                "id": got["ids"][0][i],
                "distance": got["distances"][0][i],
                "document": got["documents"][0][i],
            }
        )
    return rows


QUESTION = "How long do I have to return an unopened CD?"
hits = retrieve(QUESTION)
for row in hits:
    print(f"{row['distance']:.3f}  {row['id']}")
    print(row["document"].splitlines()[0])
    print()


0.772  policy_returns.md
# Returns

1.071  policy_warranty.md
# Warranty on physical media

1.140  policy_damaged_media.md
# Damaged media



`policy_returns.md` should be first. Distance is smaller-is-nearer. If a ticket about a warped vinyl outranks the policy, say so — the question used "CD" and the ticket used "vinyl."

### One generate

Stuff the retrieved text into a user message. Tell the model to use only those documents. If they do not contain the answer, it should say it does not know.


In [5]:
def answer(question: str, k: int = 3):
    hits = retrieve(question, k=k)
    packed = "\n\n".join(f"# {row['id']}\n{row['document']}" for row in hits)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer using only the documents. "
                    "If they do not contain the answer, say you do not know. "
                    "Do not invent a number, a date, or a phone number."
                ),
            },
            {
                "role": "user",
                "content": "Documents:\n\n" + packed + "\n\nQuestion: " + question,
            },
        ],
        max_completion_tokens=160,
        reasoning_effort="none",
    )
    return hits, response.choices[0].message.content, response.usage.prompt_tokens


hits, text, prompt_tokens = answer(QUESTION)
print("used:", [row["id"] for row in hits])
print("prompt_tokens:", prompt_tokens)
print()
print(text)


used: ['policy_returns.md', 'policy_warranty.md', 'policy_damaged_media.md']
prompt_tokens: 301

You can return an unopened CD within **30 days of the invoice date**.


That sentence should be the 30-day rule, from the file you already read.

### A question the folder cannot answer

Same function. The corpus has no 2014 plan and no executive's personal number.


In [6]:
UNANSWERABLE = "What is the CEO's personal mobile number?"
hits_u, text_u, tokens_u = answer(UNANSWERABLE)
print("used:", [row["id"] for row in hits_u])
print("prompt_tokens:", tokens_u)
print()
print(text_u)


used: ['policy_escalations.md', 'policy_support_reps.md', 'policy_privacy.md']
prompt_tokens: 306

I do not know. The provided documents only say that the shop does not publish the manager’s personal phone number, and they do not mention the CEO or any personal mobile number.


If it said it does not know, retrieval did its job and the prompt held. If it invented a dollar figure, you have a chatbot that happened to search first. Module 12 is one response to that. "I do not know" is a valid outcome. It is not a broken cell.

## 3. Observe

Five questions. Retrieve only — no generate. Print the top file. You should be able to predict two misses before the cell runs.


In [7]:
questions = [
    "How long do I have to return an unopened CD?",
    "When is support staffed?",
    "Do gift cards expire?",
    "What is Chinook's revenue target for 2014?",
    "What is the CEO's personal mobile number?",
]
print(f"{'top file':<36} {'dist':>6}  question")
for q in questions:
    row = retrieve(q, k=1)[0]
    print(f"{row['id']:<36} {row['distance']:6.3f}  {q}")


top file                               dist  question


policy_returns.md                     0.772  How long do I have to return an unopened CD?


policy_support_reps.md                0.944  When is support staffed?


policy_gift_cards.md                  0.584  Do gift cards expire?


ticket_04_canada_free_shipping.md     1.498  What is Chinook's revenue target for 2014?


policy_escalations.md                 1.394  What is the CEO's personal mobile number?


The last two will still return *a* file. Nearest is not the same as relevant. The generate step is what has to refuse.

Things to notice:

- The embedding is the only new API. Chroma did not call OpenAI.
- `prompt_tokens` on the generate is the three files plus the question, not the whole folder. That is module 05, applied to retrieval.
- A top hit on an unanswerable question is not a bug in Chroma. It is why "always answer from context" is an incomplete instruction.

LlamaIndex would wrap `embed` + `add` + `query` in an `Index`. Pinecone would put the same vectors on someone else's machine. The loop above them does not change. We stay on Chroma.

## 4. Challenge

Same `answer` function (or the same two steps written out). A new question:

> What is the student discount on physical items?

Bind:

- `hits` — the retrieve rows
- `text` — the generated sentence

The next cell checks that a sentence came back and that it mentions **10**. It does not score the wording. We will look at `policy_student_discount.md` in the debrief.


In [ ]:
# hits, text = ...


In [ ]:
assert hits and len(hits) >= 1, "hits should be the retrieve rows"
assert text and str(text).strip(), "text should be the generated sentence"
assert "10" in str(text), "the policy is 10 percent off physical items"
print("looks good")
